# Step 11: Round 1g Relation-Chain Minimal Rerun

这个 notebook 只做一件事：在不改模型、不改 prompt scaffold、不改 scoring 的前提下，
对已经 reroute 的 `relation_chain_bridge` targets 做最小 subtype-aware rerun。

固定：

- `Round 1b` prompt scaffold
- model / decoding params
- scoring / answer extraction

只改变：

- relation-chain targets 的 relevant / irrelevant source routing

当前只跑：

- `wiki_dev_2639`
- `wiki_dev_1379`


In [1]:
import csv
import gc
import json
import os
import re
import string
from pathlib import Path

try:
    import torch
except ImportError:
    torch = None

try:
    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
except ImportError:
    AutoModelForCausalLM = None
    AutoTokenizer = None
    BitsAndBytesConfig = None


## 1. 配置路径、模型与运行参数


In [2]:
# ── 路径与模型配置 ──
PROJECT_ROOT_OVERRIDE = ''


def candidate_roots():
    candidates = []
    if PROJECT_ROOT_OVERRIDE.strip():
        candidates.append(Path(PROJECT_ROOT_OVERRIDE).expanduser())

    env_root = os.environ.get('SELECT_TRANSFER_ROOT', '').strip()
    if env_root:
        candidates.append(Path(env_root).expanduser())

    cwd = Path.cwd().resolve()
    candidates.extend([
        cwd,
        cwd.parent,
        cwd / '2026_SelectTransfer',
        Path('/content/2026_SelectTransfer'),
        Path('/workspace/2026_SelectTransfer'),
        Path('/root/2026_SelectTransfer'),
        Path('/kaggle/working/2026_SelectTransfer'),
    ])

    seen = set()
    ordered = []
    for candidate in candidates:
        if candidate in seen:
            continue
        seen.add(candidate)
        ordered.append(candidate)
    return ordered


def detect_project_root():
    checked = []
    for candidate in candidate_roots():
        checked.append(str(candidate))
        if candidate.exists() and (candidate / 'pilot').exists() and (candidate / 'results').exists():
            return candidate
    raise FileNotFoundError(
        'Could not locate project root. Set PROJECT_ROOT_OVERRIDE or SELECT_TRANSFER_ROOT to the uploaded 2026_SelectTransfer directory. Checked: ' + ' | '.join(checked)
    )


PROJECT_ROOT = detect_project_root()
ARTIFACTS_DIR = PROJECT_ROOT / 'artifacts'
RESULTS_DIR = PROJECT_ROOT / 'results' / '10_round1g_run'
RAW_OUT_DIR = RESULTS_DIR / 'raw_outputs'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
RAW_OUT_DIR.mkdir(parents=True, exist_ok=True)

PAIRING_PATH = PROJECT_ROOT / 'pilot' / 'pairing_table.csv'
SUBSET_PATH = PROJECT_ROOT / 'results' / '10_round1g_prep' / 'relation_chain_minirun_subset.csv'
SAMPLED_JSON_PATH = PROJECT_ROOT / 'results' / '01_sampling' / 'sampled_20_full.json'
EXPANDED_JSON_PATH = PROJECT_ROOT / 'results' / '02_hotpotqa_comparison_expansion' / 'candidate_batch_filtered_full.json'

MODEL_ID = 'Qwen/Qwen3.5-9B'
FALLBACK_MODEL_ID = 'Qwen/Qwen3.5-4B'
HF_TOKEN = os.environ.get('HF_TOKEN', '')

RUN_GENERATION = True

USE_API = False
API_KEY = os.environ.get('OPENAI_API_KEY', '')
API_BASE_URL = os.environ.get('OPENAI_API_BASE', '')
API_MODEL = os.environ.get('PILOT_API_MODEL', 'gpt-4o-mini')

USE_4BIT = False
ENABLE_THINKING = False
MAX_NEW_TOKENS = 1600
DO_SAMPLE = False

REQUIRED_INPUTS = {
    'working pairing table': PAIRING_PATH,
    'Round 1g minirun subset': SUBSET_PATH,
    'Round 1 sampled payload json': SAMPLED_JSON_PATH,
    'comparison expansion payload json': EXPANDED_JSON_PATH,
    'attribute_bridge episodic trace': ARTIFACTS_DIR / 'hp_bridge_set_01' / 'episodic_trace.md',
    'attribute_bridge consolidation': ARTIFACTS_DIR / 'hp_bridge_set_01' / 'cross_episode_consolidation.md',
    'relation_chain episodic trace': ARTIFACTS_DIR / 'hp_relation_chain_bridge_set_01' / 'episodic_trace.md',
    'relation_chain consolidation': ARTIFACTS_DIR / 'hp_relation_chain_bridge_set_01' / 'cross_episode_consolidation.md',
}

missing_inputs = {name: str(path) for name, path in REQUIRED_INPUTS.items() if not path.exists()}
if missing_inputs:
    missing_lines = [f'- {name}: {path}' for name, path in missing_inputs.items()]
    raise FileNotFoundError('Detected project root but some required inputs are missing:\n' + '\n'.join(missing_lines))

print('PROJECT_ROOT =', PROJECT_ROOT)
print('RESULTS_DIR =', RESULTS_DIR)
print('MODEL_ID =', MODEL_ID)
print('RUN_GENERATION =', RUN_GENERATION)
print('USE_API =', USE_API)
print('required inputs checked =', len(REQUIRED_INPUTS))
print('torch import available =', torch is not None)
if torch is not None and torch.cuda.is_available():
    print('CUDA device =', torch.cuda.get_device_name(0))
    print('CUDA bf16 supported =', torch.cuda.is_bf16_supported())
    total_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    print('CUDA total memory (GB) =', round(total_gb, 2))
print('transformers import available =', AutoTokenizer is not None)


PROJECT_ROOT = /root/2026_SelectTransfer
RESULTS_DIR = /root/2026_SelectTransfer/results/05_round1b_run
MODEL_ID = Qwen/Qwen3.5-9B
FALLBACK_MODEL_ID = Qwen/Qwen3.5-4B
RUN_GENERATION = True
USE_API = False
torch import available = True
CUDA device = NVIDIA GeForce RTX 4090
CUDA bf16 supported = True
CUDA total memory (GB) = 47.37
transformers import available = True


## 2. 加载 working pairing、Round 1g subset 与 artifacts


In [3]:
def read_csv(path):
    with path.open(newline='', encoding='utf-8') as f:
        return list(csv.DictReader(f))


def load_json(path):
    return json.loads(path.read_text(encoding='utf-8'))


pairing_rows = read_csv(PAIRING_PATH)
subset_rows = read_csv(SUBSET_PATH)
subset_ids = {row['target_task_id'] for row in subset_rows}

sampled_rows = load_json(SAMPLED_JSON_PATH)
expanded_rows = load_json(EXPANDED_JSON_PATH)
all_payload_rows = sampled_rows + expanded_rows
all_payload = {row['task_id']: row for row in all_payload_rows}

filtered_pairing = [row for row in pairing_rows if row['target_task_id'] in subset_ids]
missing_pairing_ids = sorted(subset_ids - {row['target_task_id'] for row in filtered_pairing})
if missing_pairing_ids:
    raise FileNotFoundError('Subset targets missing from working pairing table: ' + ', '.join(missing_pairing_ids))

missing_payload_ids = sorted(subset_ids - set(all_payload.keys()))
if missing_payload_ids:
    raise FileNotFoundError('Subset targets missing from payload json inputs: ' + ', '.join(missing_payload_ids))

artifact_source_set_ids = sorted({
    row['relevant_source_set_id'] for row in filtered_pairing
} | {
    row['irrelevant_source_set_id'] for row in filtered_pairing
})

artifact_cache = {}
for source_set_id in artifact_source_set_ids:
    base = ARTIFACTS_DIR / source_set_id
    for artifact_type, fname in [
        ('episodic_trace', 'episodic_trace.md'),
        ('cross_episode_consolidation', 'cross_episode_consolidation.md'),
    ]:
        path = base / fname
        artifact_cache[(source_set_id, artifact_type)] = path.read_text(encoding='utf-8')

print('subset targets:', sorted(subset_ids))
print('pairing rows used:', len(filtered_pairing))
print('artifact source sets:', artifact_source_set_ids)
print('artifact cache entries:', len(artifact_cache))


smoke targets: 6
pairing rows used: 6
artifact cache entries: 4


## 3. Round 1b prompt scaffold 与解析规则


In [4]:
def build_context_paragraphs(raw_context):
    context = raw_context.get('context', {})

    if isinstance(context, dict):
        titles = context.get('title', []) or []
        sentences = context.get('sentences', []) or []
        paragraphs = []
        for title, sents in zip(titles, sentences):
            text = ' '.join(str(s) for s in sents)
            paragraphs.append(f"### {title}\n{text}")
        return "\n\n".join(paragraphs)

    if isinstance(context, str):
        try:
            parsed = json.loads(context)
        except json.JSONDecodeError:
            return context.strip()

        paragraphs = []
        for item in parsed:
            if not isinstance(item, list) or len(item) != 2:
                continue
            title, sents = item
            text = ' '.join(str(s) for s in sents)
            paragraphs.append(f"### {title}\n{text}")
        return "\n\n".join(paragraphs)

    return str(context)


def assemble_prompt(target_task, condition, artifact_content=None):
    context = build_context_paragraphs(target_task['raw'])
    question = target_task['question']

    base_header = (
        'You are a question-answering agent. '
        'Your task is to answer a multi-hop reasoning question '
        'using the provided context paragraphs.'
    )

    context_section = f"## Context\n\n{context}"
    question_section = f"## Question\n\n{question}"

    if condition == 'no_memory':
        memory_section = ''
    else:
        memory_section = (
            '## Past Experience\n\n'
            'The following notes summarize patterns from previously solved tasks '
            'that may or may not be relevant to the current question. '
            'Use them only if they help your reasoning — do not force-apply them.\n\n'
            f'{artifact_content}'
        )

    instructions_section = (
        '## Instructions\n\n'
        '- Read all context paragraphs carefully.\n'
        '- Work through the reasoning chain explicitly before deciding the answer.\n'
        '- In ## Reasoning, write 3 to 6 short bullet points grounded in the provided context.\n'
        '- If past experience is shown, either use it explicitly or state briefly why it is not useful here.\n'
        '- Keep the reasoning concise and evidence-grounded.\n'
        '- In ## Final Answer, give only the final short answer phrase.'
    )

    reasoning_section = '## Reasoning'
    final_answer_section = '## Final Answer'

    parts = [base_header, '', context_section, '', question_section]
    if memory_section:
        parts += ['', memory_section]
    parts += ['', instructions_section, '', reasoning_section, '', final_answer_section]
    return "\n\n".join(parts)


def extract_final_answer(model_output):
    if '## Final Answer' in model_output:
        tail = model_output.split('## Final Answer')[-1].strip().splitlines()
        for line in tail:
            line = line.strip()
            if line:
                return line
        return ''
    lines = [l.strip() for l in model_output.strip().split('\n') if l.strip()]
    return lines[-1] if lines else ''


def reasoning_present(model_output):
    return int('## Reasoning' in model_output)


def final_answer_present(model_output):
    return int('## Final Answer' in model_output)


def memory_reference_type(model_output):
    low = model_output.lower()
    if 'past experience' in low and ('not useful' in low or 'not relevant' in low or 'ignore' in low):
        return 'explicit_reject'
    if 'past experience' in low or 'the notes' in low or 'the memory' in low or 'based on the pattern' in low:
        return 'explicit_use'
    return 'implicit_or_none'


def normalize_answer(text):
    text = text.lower()
    text = re.sub(r'\b(a|an|the)\b', ' ', text)
    text = text.translate(str.maketrans('', '', string.punctuation))
    text = ' '.join(text.split())
    return text


def compute_em(pred, gold):
    return int(normalize_answer(pred) == normalize_answer(gold))


def compute_f1(pred, gold):
    pred_tokens = set(normalize_answer(pred).split())
    gold_tokens = set(normalize_answer(gold).split())
    if not pred_tokens or not gold_tokens:
        return 0.0
    common = pred_tokens & gold_tokens
    if not common:
        return 0.0
    precision = len(common) / len(pred_tokens)
    recall = len(common) / len(gold_tokens)
    return 2 * precision * recall / (precision + recall)

print('prompt scaffold ready')


prompt scaffold ready


## 4. 本地模型加载


In [5]:
def infer_torch_dtype():
    if torch is None:
        raise ImportError('torch is not available.')
    if torch.cuda.is_available():
        if torch.cuda.is_bf16_supported():
            return torch.bfloat16
        return torch.float16
    return torch.float32


def load_local_model(model_id):
    if torch is None:
        raise ImportError('torch is not available.')
    if AutoTokenizer is None or AutoModelForCausalLM is None:
        raise ImportError('transformers is not available.')

    token = HF_TOKEN or None
    tokenizer = AutoTokenizer.from_pretrained(model_id, token=token)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    model_kwargs = {'device_map': 'auto', 'token': token}
    torch_dtype = infer_torch_dtype()
    if USE_4BIT:
        model_kwargs['quantization_config'] = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch_dtype if torch_dtype != torch.float32 else torch.float16,
            bnb_4bit_quant_type='nf4',
            bnb_4bit_use_double_quant=True,
        )
    else:
        model_kwargs['torch_dtype'] = torch_dtype

    model = AutoModelForCausalLM.from_pretrained(model_id, **model_kwargs)
    model.eval()
    return tokenizer, model


tokenizer = None
model = None

if RUN_GENERATION and not USE_API:
    tokenizer, model = load_local_model(MODEL_ID)
    print(f'Loaded local model: {MODEL_ID}')
elif RUN_GENERATION and USE_API:
    print(f'Will use API backend: {API_MODEL}')
else:
    print('RUN_GENERATION = False -> dry run only')


`torch_dtype` is deprecated! Use `dtype` instead!


Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Loading weights:   0%|          | 0/427 [00:00<?, ?it/s]

Loaded local model: Qwen/Qwen3.5-9B


## 5. 构造 Round 1g run plan


In [6]:
run_plan = []
for row in filtered_pairing:
    tid = row['target_task_id']

    run_plan.append({
        'run_id': f'r1g_no_memory_{tid}',
        'target_task_id': tid,
        'split': 'none',
        'condition': 'no_memory',
        'source_set_id': 'none',
        'artifact_type': 'none',
    })

    for condition in ['episodic_trace', 'cross_episode_consolidation']:
        for split in ['relevant', 'irrelevant']:
            source_set_id = row['relevant_source_set_id'] if split == 'relevant' else row['irrelevant_source_set_id']
            run_plan.append({
                'run_id': f'r1g_{condition}_{tid}_{split}',
                'target_task_id': tid,
                'split': split,
                'condition': condition,
                'source_set_id': source_set_id,
                'artifact_type': condition,
            })

print('planned runs:', len(run_plan))
for row in run_plan:
    print(row)


planned runs: 36
{'run_id': 'r1b_no_memory_wiki_dev_8896_relevant', 'target_task_id': 'wiki_dev_8896', 'split': 'relevant', 'condition': 'no_memory', 'source_set_id': 'hp_comparison_set_01', 'artifact_type': 'none'}
{'run_id': 'r1b_no_memory_wiki_dev_8896_irrelevant', 'target_task_id': 'wiki_dev_8896', 'split': 'irrelevant', 'condition': 'no_memory', 'source_set_id': 'hp_bridge_set_01', 'artifact_type': 'none'}
{'run_id': 'r1b_episodic_trace_wiki_dev_8896_relevant', 'target_task_id': 'wiki_dev_8896', 'split': 'relevant', 'condition': 'episodic_trace', 'source_set_id': 'hp_comparison_set_01', 'artifact_type': 'episodic_trace'}
{'run_id': 'r1b_episodic_trace_wiki_dev_8896_irrelevant', 'target_task_id': 'wiki_dev_8896', 'split': 'irrelevant', 'condition': 'episodic_trace', 'source_set_id': 'hp_bridge_set_01', 'artifact_type': 'episodic_trace'}
{'run_id': 'r1b_cross_episode_consolidation_wiki_dev_8896_relevant', 'target_task_id': 'wiki_dev_8896', 'split': 'relevant', 'condition': 'cross_ep

## 6. 生成函数与主循环


In [ ]:
def build_chat_text(prompt_text):
    messages = [{'role': 'user', 'content': prompt_text}]
    try:
        return tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=ENABLE_THINKING,
        )
    except TypeError:
        return tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )


def get_model_device(model_obj):
    return next(model_obj.parameters()).device


def generate_local(prompt_text):
    if torch is None:
        raise ImportError('torch is not available.')
    chat_text = build_chat_text(prompt_text)
    model_inputs = tokenizer([chat_text], return_tensors='pt')
    model_device = get_model_device(model)
    model_inputs = {k: v.to(model_device) for k, v in model_inputs.items()}
    with torch.inference_mode():
        generated_ids = model.generate(
            **model_inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=DO_SAMPLE,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    output_ids = generated_ids[0][model_inputs['input_ids'].shape[1]:]
    text = tokenizer.decode(output_ids, skip_special_tokens=True).strip()
    if not ENABLE_THINKING:
        text = text.replace('<think>', '').replace('</think>', '').strip()
    token_usage = len(model_inputs['input_ids'][0]) + len(output_ids)
    return text, token_usage


def generate_api(prompt_text):
    import httpx
    headers = {'Authorization': f'Bearer {API_KEY}', 'Content-Type': 'application/json'}
    payload = {
        'model': API_MODEL,
        'messages': [{'role': 'user', 'content': prompt_text}],
        'max_tokens': MAX_NEW_TOKENS,
        'temperature': 0.0,
    }
    base = API_BASE_URL.rstrip('/') if API_BASE_URL else 'https://api.openai.com/v1'
    resp = httpx.post(f'{base}/chat/completions', json=payload, headers=headers, timeout=180)
    resp.raise_for_status()
    data = resp.json()
    text = data['choices'][0]['message']['content'].strip()
    usage = data.get('usage', {})
    total_tokens = usage.get('total_tokens', 0)
    return text, total_tokens


def generate(prompt_text):
    if USE_API:
        return generate_api(prompt_text)
    return generate_local(prompt_text)


results = []
for i, run in enumerate(run_plan):
    tid = run['target_task_id']
    condition = run['condition']
    split = run['split']
    source_set_id = run['source_set_id']
    target_task = all_payload[tid]
    gold = target_task['answer']

    artifact_content = None
    if condition != 'no_memory':
        artifact_content = artifact_cache[(source_set_id, run['artifact_type'])]

    prompt = assemble_prompt(target_task, condition, artifact_content)
    raw_output = ''
    pred_answer = ''
    token_usage = 0
    failure_status = 'ok'

    if RUN_GENERATION:
        try:
            raw_output, token_usage = generate(prompt)
            pred_answer = extract_final_answer(raw_output)
        except Exception as e:
            failure_status = f'error: {str(e)[:200]}'
    else:
        failure_status = 'dry_run'

    em = compute_em(pred_answer, gold) if pred_answer else 0
    f1 = round(compute_f1(pred_answer, gold), 4) if pred_answer else 0.0
    rp = reasoning_present(raw_output) if raw_output else 0
    fp = final_answer_present(raw_output) if raw_output else 0
    mrt = memory_reference_type(raw_output) if raw_output else 'implicit_or_none'
    parse_success = int(bool(pred_answer))

    results.append({
        'run_id': run['run_id'],
        'target_task_id': tid,
        'split': split,
        'condition': condition,
        'source_set_id': source_set_id,
        'memory_attached': 'false' if condition == 'no_memory' else 'true',
        'em': em,
        'f1': f1,
        'token_usage': token_usage,
        'failure_status': failure_status,
        'pred_answer': pred_answer,
        'gold_answer': gold,
        'prompt_chars': len(prompt),
        'reasoning_present': rp,
        'final_answer_present': fp,
        'memory_reference_type': mrt,
        'parse_success': parse_success,
        'raw_output': raw_output,
        'prompt_text': prompt,
    })

    if raw_output or failure_status == 'dry_run':
        raw_path = RAW_OUT_DIR / f"{run['run_id']}.md"
        with raw_path.open('w', encoding='utf-8') as f:
            f.write(f"# {run['run_id']}\n\n")
            f.write(f"- target_task_id: {tid}\n")
            f.write(f"- split: {split}\n")
            f.write(f"- condition: {condition}\n")
            f.write(f"- source_set_id: {source_set_id}\n")
            f.write(f"- gold_answer: {gold}\n")
            f.write(f"- pred_answer: {pred_answer}\n")
            f.write(f"- em: {em}\n")
            f.write(f"- f1: {f1}\n")
            f.write(f"- token_usage: {token_usage}\n")
            f.write(f"- prompt_chars: {len(prompt)}\n")
            f.write(f"- reasoning_present: {rp}\n")
            f.write(f"- final_answer_present: {fp}\n")
            f.write(f"- memory_reference_type: {mrt}\n")
            f.write(f"- parse_success: {parse_success}\n")
            f.write(f"- failure_status: {failure_status}\n\n")
            f.write('---\n\n')
            f.write(f"## Prompt\n\n```\n{prompt}\n```\n\n")
            f.write(f"## Raw Model Output\n\n```\n{raw_output}\n```\n")

    if (i + 1) % 5 == 0 or i == len(run_plan) - 1:
        print(f'progress: {i + 1}/{len(run_plan)}')

print('total results:', len(results))


progress: 12/36
progress: 24/36


## 7. 写出结果文件


In [ ]:
detail_fieldnames = [
    'run_id', 'target_task_id', 'split', 'condition', 'source_set_id',
    'memory_attached', 'em', 'f1', 'token_usage', 'failure_status',
    'pred_answer', 'gold_answer', 'prompt_chars', 'reasoning_present',
    'final_answer_present', 'memory_reference_type', 'parse_success', 'raw_output'
]

detail_path = RESULTS_DIR / 'round1g_relation_chain_results_detail.csv'
with detail_path.open('w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=detail_fieldnames)
    writer.writeheader()
    for r in results:
        writer.writerow({k: r[k] for k in detail_fieldnames})
print('detail ->', detail_path)

summary_fieldnames = [
    'run_id', 'target_task_id', 'split', 'condition', 'source_set_id',
    'memory_attached', 'em', 'f1', 'token_usage', 'failure_status',
    'reasoning_present', 'final_answer_present', 'memory_reference_type', 'parse_success'
]
summary_path = RESULTS_DIR / 'round1g_relation_chain_results.csv'
with summary_path.open('w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=summary_fieldnames)
    writer.writeheader()
    for r in results:
        writer.writerow({k: r[k] for k in summary_fieldnames})
print('summary ->', summary_path)


## 8. Quick sanity check


In [ ]:
for r in results[:6]:
    print(r['run_id'], r['failure_status'], r['reasoning_present'], r['final_answer_present'], r['memory_reference_type'])

if model is not None:
    gc.collect()
    if torch is not None and torch.cuda.is_available():
        torch.cuda.empty_cache()
